In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

## 推理指标

In [1]:
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

jsonl_path = "/hdd/wangty/diffuser_workdir/result/origin_zjppt_c3-4.jsonl"

y_true = []
y_pred = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sample = json.loads(line)

        if "label" not in sample or "predict" not in sample:
            continue

        y_true.append(int(sample["label"]))
        y_pred.append(int(sample["predict"]))

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Num samples: {len(y_true)}")
print(f"Accuracy: {acc:.6f}")
print(f"Macro F1: {macro_f1:.6f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=6))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Num samples: 336
Accuracy: 0.842262
Macro F1: 0.725757

Classification Report:
              precision    recall  f1-score   support

           0   0.640000  0.477612  0.547009        67
           1   0.877622  0.933086  0.904505       269

    accuracy                       0.842262       336
   macro avg   0.758811  0.705349  0.725757       336
weighted avg   0.830239  0.842262  0.833218       336

Confusion Matrix:
[[ 32  35]
 [ 18 251]]


In [2]:
import json
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

jsonl_path = "/hdd/wangty/diffuser_workdir/result/gen_zjppt_c3-4.jsonl"

y_true = []
y_pred = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sample = json.loads(line)

        if "label" not in sample or "predict" not in sample:
            continue

        y_true.append(int(sample["label"]))
        y_pred.append(int(sample["predict"]))

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Num samples: {len(y_true)}")
print(f"Accuracy: {acc:.6f}")
print(f"Macro F1: {macro_f1:.6f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=6))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Num samples: 336
Accuracy: 0.630952
Macro F1: 0.503976

Classification Report:
              precision    recall  f1-score   support

           0   0.212121  0.313433  0.253012        67
           1   0.805907  0.710037  0.754941       269

    accuracy                       0.630952       336
   macro avg   0.509014  0.511735  0.503976       336
weighted avg   0.687503  0.630952  0.654854       336

Confusion Matrix:
[[ 21  46]
 [ 78 191]]


## 图像指标

In [2]:
import json

# 图像列表
with open("/hdd/wangty/new_task/LLaMA-Factory/task/dataset/id/id_split.txt","r") as f:
    train_id,val_id,test_id = eval(f.read())
id_list=train_id+val_id+test_id
print(len(id_list))
img_list1=[]
img_list2=[]
img_path="/hdd/wangty/diffuser_workdir/gen_img/2e-5_12ep/tra_topk_32"
origin_path='/mnt/nvme_share/wangty/img/tra_256'
for id in test_id:
    path1=f'{origin_path}/{id}/3-4.png'
    path2=f'{img_path}/{id}.png'
    img_list1.append(path1)
    img_list2.append(path2)

2231


In [ ]:
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from pytorch_fid.inception import InceptionV3
from pytorch_fid.fid_score import calculate_frechet_distance
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.covariance import LedoitWolf

device = "cuda" if torch.cuda.is_available() else "cpu"

# FID默认使用768维特征
dims = 768
block_idx = InceptionV3.BLOCK_INDEX_BY_DIM[dims]
model = InceptionV3([block_idx]).to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

In [9]:
def get_activations(img_list, model, batch_size=32, device="cuda"):
    acts = []

    for i in tqdm(range(0, len(img_list), batch_size)):
        batch_paths = img_list[i:i + batch_size]
        imgs = []

        for path in batch_paths:
            img = Image.open(path).convert("RGB")
            img = transform(img)
            imgs.append(img)

        imgs = torch.stack(imgs, dim=0).to(device)

        with torch.no_grad():
            pred = model(imgs)[0]

        # 不管输出是不是 [N,C,1,1]，都统一池化成 [N,C,1,1]
        pred = F.adaptive_avg_pool2d(pred, output_size=(1, 1))

        # 再变成 [N, C]
        pred = pred.squeeze(-1).squeeze(-1).cpu().numpy()

        acts.append(pred)

    acts = np.concatenate(acts, axis=0)
    return acts
from sklearn.covariance import LedoitWolf
import numpy as np

def calculate_activation_statistics(img_list, model, batch_size=32, dims=192, device="cuda"):
    act = get_activations(img_list, model, batch_size=batch_size, device=device)
    print("act shape:", act.shape)  # 应该是 [N, dims]

    mu = np.mean(act, axis=0)
    sigma = LedoitWolf().fit(act).covariance_
    return mu, sigma

In [10]:
mu1, sigma1 = calculate_activation_statistics(img_list1, model, batch_size=32, dims=dims, device=device)
mu2, sigma2 = calculate_activation_statistics(img_list2, model, batch_size=32, dims=dims, device=device)

fid_value = calculate_frechet_distance(mu1, sigma1, mu2, sigma2)
print("FID:", fid_value)

100%|██████████| 11/11 [00:01<00:00,  5.64it/s]


act shape: (336, 768)


100%|██████████| 11/11 [01:22<00:00,  7.46s/it]


act shape: (336, 768)
FID: 0.27699095010757446
